# Run on Temporal and reconnect after a worker restart

Submit a durable job, disconnect the client, then retrieve the result. Optionally crash this notebook’s own worker.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/06_durable.ipynb)

Run these cells in order in **Google Colab**. Everything runs in its cloud runtime:
no repository checkout or laptop installation. A CPU runtime is enough.
Model calls use your provider account. Clear outputs before sharing a saved copy.

## Install and choose a model

Keep the defaults for a first run. To switch providers, change `MODEL`:
- OpenAI: `openai/gpt-5.4-mini` with `OPENAI_API_KEY`.
- Anthropic: `anthropic/claude-sonnet-4-6` with `ANTHROPIC_API_KEY`.
- OpenRouter: `openrouter/anthropic/claude-sonnet-4.6` with `OPENROUTER_API_KEY`.

LiteLLM's Python SDK handles provider translation; no gateway is required.
Optional: set `API_BASE` to your gateway URL and use its exact model alias
with `LITELLM_API_KEY`. Leave `API_BASE` empty for direct provider access.

For Gemini, Groq, Mistral, DeepSeek, Together AI, xAI, Azure, Bedrock,
Vertex AI, or Ollama, see the [model setup guide](https://github.com/BerriAI/liteagents/blob/main/docs/models.md).
Cloud authentication and provider-specific environment variables must be configured
in this runtime before running the agent.

In [ ]:
import os

HARNESS = os.environ.get("LITEAGENTS_HARNESS", "deepagents")
MODEL = os.environ.get("LITEAGENTS_MODEL", "openai/gpt-5.4-mini")
API_BASE = os.environ.get("LITEAGENTS_API_BASE", "")  # Optional gateway URL.

Available harnesses: `deepagents`, `pydantic-ai`, `claude-sdk`, `codex`,
`opencode-v1`, `opencode-v2`. Rerun the install cell after changing your selection.
Packages are reused within this runtime; a fresh Colab runtime needs its own install.
OpenCode is installed only when selected.

This preview installs from a GitHub release wheel because the PyPI name currently
belongs to another package. It does not clone the repository.

In [ ]:
# @title Install selected integrations
import shutil
import subprocess
import sys

selected_harnesses = [HARNESS]
extras = sorted(set(selected_harnesses) | {"temporal"})
release = "https://github.com/BerriAI/liteagents/releases/download/v0.3.0a3"
package = f"liteagents[{','.join(extras)}] @ {release}/liteagents-0.3.0a3-py3-none-any.whl"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", package,
    "-c", f"{release}/constraints-tested.txt",
])
if any(h.startswith("opencode-") for h in selected_harnesses):
    if shutil.which("opencode") is None:
        subprocess.check_call(["npm", "install", "-g", "opencode-ai@1.18.29"])
    subprocess.check_call(["opencode", "--version"])
print("Ready:", ", ".join(selected_harnesses))

## Add your API key

In Colab, open the **key icon → Secrets**, add the key named above, and enable
notebook access. Or enter it in the hidden prompt below. The key stays out of your
code and saved outputs. If installation asks for a runtime restart,
restart once and run the cells again.

In [ ]:
# @title Connect your provider
from getpass import getpass

KEY_NAME = "LITELLM_API_KEY" if API_BASE else {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "together_ai": "TOGETHERAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "xai": "XAI_API_KEY",
    "azure": "AZURE_API_KEY",
}.get(MODEL.split("/", 1)[0])
API_KEY = None
if KEY_NAME:
    API_KEY = os.environ.get(KEY_NAME)
    if not API_KEY:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                API_KEY = userdata.get(KEY_NAME)
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    API_KEY = API_KEY or getpass(f"{KEY_NAME}: ")
    if not API_KEY:
        raise ValueError(f"Provide {KEY_NAME} before running the agent.")
    os.environ[KEY_NAME] = API_KEY
else:
    print("Using provider credentials from the runtime; see the model setup guide.")
os.environ["LITEAGENTS_MODEL"] = MODEL
if API_BASE:
    os.environ["LITEAGENTS_API_BASE"] = API_BASE
MODEL_KWARGS = {"api_base": API_BASE, "api_key": API_KEY} if API_BASE else {}

## Create your profile

This is the SDK interface. The following cells change this profile to demonstrate
one feature. A temporary workspace keeps each run's files separate.

In [ ]:
import asyncio
import tempfile
from pathlib import Path

from liteagents import LiteAgentClient, LiteAgentOptions, ProfileOptions

profile = ProfileOptions(
    harness=HARNESS,
    model=MODEL,
    model_kwargs=MODEL_KWARGS,
    tools=[],
    max_turns=10,
    system_prompt="Use the requested tools and report their actual results. Be concise.",
)
workspace_root = Path(os.environ.get(
    "LITEAGENTS_NOTEBOOK_WORKSPACE", Path(tempfile.gettempdir()) / "liteagents-notebooks"
))
workspace_root.mkdir(parents=True, exist_ok=True)
workspace = Path(tempfile.mkdtemp(prefix="run-", dir=workspace_root)).resolve()
print("Workspace:", workspace)

In [ ]:
DELAY_SECONDS = 30
RUN_CRASH_DEMO = False  # True kills only the worker started by this notebook, then restarts it.
USE_TEMPORAL = True

## Start a demo Temporal service

This cell downloads and starts Temporal **inside the Colab runtime**.
No terminal or separate account is needed. A runtime reset deletes local checkpoints;
persistent deployments need an external [Temporal service](https://github.com/BerriAI/liteagents/blob/main/docs/self-hosting.md).
Set `LITEAGENTS_TEMPORAL_ADDRESS` to use an existing development service instead.

In [ ]:
temporal_environment = globals().get("temporal_environment")
temporal_address = os.environ.get("LITEAGENTS_TEMPORAL_ADDRESS")
if USE_TEMPORAL:
    from temporalio.testing import WorkflowEnvironment

    if temporal_environment is not None:
        temporal_address = temporal_environment.client.service_client.config.target_host
    elif not temporal_address:
        temporal_environment = await WorkflowEnvironment.start_local(
            ui=False,
            dev_server_existing_path=shutil.which("temporal"),
            dev_server_database_filename=str(workspace / "temporal.sqlite"),
        )
        temporal_address = temporal_environment.client.service_client.config.target_host
    print("Temporal:", temporal_address)

In [ ]:
from liteagents import TemporalOptions

profile.tools = ["save_receipt", "slow_check"]
profile.temporal = TemporalOptions(
    address=temporal_address,
    checkpoint_path=str(workspace / "checkpoints.sqlite"),
    heartbeat_timeout_seconds=3,
    activity_timeout_seconds=180,
)
options = LiteAgentOptions(profile=profile, cwd=workspace)
worker_log = workspace / "worker.log"
worker_script = workspace / "worker.py"
worker_script.write_text(r'''
import asyncio
import os
from pathlib import Path
from typing import ClassVar
from liteagents import ProfileOptions, Tool, operation_id
from liteagents.temporal import LiteAgentWorker

class Receipt(Tool):
    name = "save_receipt"
    description = "Save the receipt once and return its tracking code."
    input_schema: ClassVar[dict] = {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    }

    def __init__(self, cwd):
        self.cwd = cwd

    async def execute(self, input):
        key = operation_id()
        assert key
        directory = self.cwd / "receipts"
        directory.mkdir(exist_ok=True)
        path = directory / key
        try:
            with path.open("x") as stream:
                stream.write("receipt-verified")
            print("Receipt saved once", flush=True)
        except FileExistsError:
            print("Reused the receipt idempotency key", flush=True)
        return "Receipt saved: receipt-verified"

class SlowCheck(Tool):
    name = "slow_check"
    description = "Check the saved receipt and report its verification result."
    input_schema: ClassVar[dict] = {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    }

    def __init__(self, delay):
        self.delay = delay

    async def execute(self, input):
        print(
            "Slow check started; this is the point to kill the worker in the crash demo.",
            flush=True,
        )
        await asyncio.sleep(self.delay)
        print("Slow check completed", flush=True)
        return "Receipt verification passed: receipt-verified"

async def main():
    profile = ProfileOptions.model_validate_json(os.environ["NOTEBOOK_WORKER_PROFILE"])
    cwd = Path(os.environ["NOTEBOOK_WORKER_CWD"])
    tools = [Receipt(cwd), SlowCheck(int(os.environ["NOTEBOOK_WORKER_DELAY"]))]
    await LiteAgentWorker(profile=profile, tools=tools, cwd=cwd).run()

asyncio.run(main())
''')
worker_command = [sys.executable, "-u", str(worker_script)]

async def start_worker():
    global worker
    previous = globals().get("worker")
    if previous is not None and previous.returncode is None:
        raise RuntimeError("A notebook worker is already running. Run the cleanup cell first.")
    with worker_log.open("a") as log:
        worker = await asyncio.create_subprocess_exec(
            *worker_command, cwd=workspace, stdout=log, stderr=asyncio.subprocess.STDOUT,
            env={**os.environ, "NOTEBOOK_WORKER_PROFILE": profile.model_dump_json(),
                 "NOTEBOOK_WORKER_CWD": str(workspace),
                 "NOTEBOOK_WORKER_DELAY": str(DELAY_SECONDS)},
        )
    print("Worker PID:", worker.pid)

async def stop_worker(*, crash=False):
    process = globals().get("worker")
    if process is not None and process.returncode is None:
        process.kill() if crash else process.terminate()
        try:
            await asyncio.wait_for(process.wait(), 10)
        except TimeoutError:
            process.kill()
            await process.wait()

## Start a worker and submit a job

The run ID is generated afresh. The submitting client closes immediately; Temporal keeps the job.
For a new profile or workspace, finish or cancel the current job and run the cleanup cell first.

In [ ]:
await start_worker()
try:
    async with LiteAgentClient(options=options) as client:
        run = await client.start_run("Call save_receipt once, then slow_check once. Report their results.")
        run_id = run.run_id
    print("Submitted:", run_id)
except BaseException:
    await stop_worker()
    raise

## Optional crash and restart

With `RUN_CRASH_DEMO=True`, this cell waits until the receipt is saved and `slow_check` has started,
then kills only the worker process it owns. The replacement worker uses the same checkpoint store.
Completed operations are reused; the interrupted check may run again.

In [ ]:
if RUN_CRASH_DEMO:
    try:
        async with asyncio.timeout(120):
            while "Slow check started" not in worker_log.read_text():
                if worker.returncode is not None:
                    raise RuntimeError("Worker exited. Inspect worker_log before retrying.")
                await asyncio.sleep(0.2)
        await stop_worker(crash=True)
        await start_worker()
        print("Restarted the worker; attaching to the same run next.")
    except BaseException:
        await stop_worker()
        raise
else:
    print("Worker continues normally. Set RUN_CRASH_DEMO=True for the recovery experiment.")

## Attach without resubmitting

`get_run()` reopens the same run. This cell always stops the notebook's worker when finished.
If you interrupt an earlier cell, run the cleanup cell at the bottom. A kernel crash can leave
its separate worker running; its PID and log are printed above.

In [ ]:
try:
    async with asyncio.timeout(240):
        async with LiteAgentClient(options=options) as client:
            attached = await client.get_run(run_id)
            result = await attached.result()
    print(result.text)
    assert "receipt-verified" in result.text
finally:
    await stop_worker()

In [ ]:
receipts = list((workspace / "receipts").glob("*"))
print("Saved receipts:", len(receipts))
print(worker_log.read_text())
assert len(receipts) == 1

## Cleanup

Only the worker and demo Temporal service created here are stopped. An external Temporal
service is left running. If a cell is interrupted, run cleanup yourself.
To cancel an unfinished workflow, attach with `client.get_run(run_id)` and call
`await run.cancel()` before stopping its worker.

This demonstrates worker recovery while the runtime and its files survive.
Colab is not a persistent production deployment.

In [ ]:
await stop_worker()
if temporal_environment is not None:
    await temporal_environment.shutdown()
    temporal_environment = None